In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
import os
from dotenv import load_dotenv
load_dotenv()

True

In [100]:
video_id = 'VJgdOMXhEj0'
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id=video_id, languages=['en', 'hi'])
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript_list)
except TranscriptsDisabled:
    print('No transcript available for this video')

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='"Speeding and..." [CLAP]', start=0.68, duration=1.272), FetchedTranscriptSnippet(text='"All lined up..."', start=1.952, duration=1.37), FetchedTranscriptSnippet(text='"Brand new season!"', start=3.322, duration=1.007), FetchedTranscriptSnippet(text='"Brand new tour for Formula 1..."', start=4.329, duration=1.899), FetchedTranscriptSnippet(text="I'm looking at the brand new car for the\nnumber one team for the world's most popular", start=6.228, duration=5.176), FetchedTranscriptSnippet(text='car racing sport, Formula 1, with the current\nworld champion driver, Max Verstappen!', start=11.404, duration=5.18), FetchedTranscriptSnippet(text='MAX: "Would you like to come and see the car?\nME: "YES"', start=16.584, duration=1.937), FetchedTranscriptSnippet(text="And we're gonna show you what it takes\nto build and drive some of the fastest and most", start=18.521, duration=5.199), FetchedTranscriptSnippet(text="expensive cars in the 

In [ ]:
from langchain_community.document_loaders import YoutubeLoader
try:
    loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=jBEOdMKAkEU", add_video_info = False
    )
    transcript = loader.load()
    print(transcript)
except Exception as e:
    print(f"Hit an error: {e}")

[Document(metadata={'source': 'jBEOdMKAkEU'}, page_content='There\'s competitors now ramping up and as you\'re familiar with BYD which is also on the west coast I think they\'re ramping up production of their electric vehicles. Uh Warren Buffett owns 10% [music] stake in that. Uh why do you laugh? BYD [laughter] is trying to compete. Why do you laugh? >> Have you seen that car? >> You know BYD was once a joke a complete laughing stock in the industry. This plain looking sedan is actually one of them. It\'s from Chinese automaker BYD. Badl looking car. It sort of looks like a Corolla or maybe a Kia Optima. Vehicles that are ready that are up to snuff in terms of quality, fit and finish, design, that remains to be seen yet. >> You don\'t see them at all as a competitor. >> No, I I don\'t I don\'t think they have a great product. In fact, if you look closely, while Tesla had Elon Musk, the visionary, charismatic billionaire founder, BYD was led by a humble man, a former government employe

In [106]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# result = splitter.create_documents([transcript])
result = splitter.split_documents(transcript)
result[:5]

[Document(metadata={'source': 'jBEOdMKAkEU'}, page_content="There's competitors now ramping up and as you're familiar with BYD which is also on the west coast I think they're ramping up production of their electric vehicles. Uh Warren Buffett owns 10% [music] stake in that. Uh why do you laugh? BYD [laughter] is trying to compete. Why do you laugh? >> Have you seen that car? >> You know BYD was once a joke a complete laughing stock in the industry. This plain looking sedan is actually one of them. It's from Chinese automaker BYD. Badl looking car. It"),
 Document(metadata={'source': 'jBEOdMKAkEU'}, page_content="from Chinese automaker BYD. Badl looking car. It sort of looks like a Corolla or maybe a Kia Optima. Vehicles that are ready that are up to snuff in terms of quality, fit and finish, design, that remains to be seen yet. >> You don't see them at all as a competitor. >> No, I I don't I don't think they have a great product. In fact, if you look closely, while Tesla had Elon Musk,

In [96]:
embedding_funtn = OllamaEmbeddings(model='embeddinggemma')
db = FAISS.from_documents(result, embedding=embedding_funtn)
# query = "What is f1?"
query = "Who is the founder of BYD?"
res = db.similarity_search(query, k=5)
res

[Document(id='56993b4e-25a6-4a31-9168-87ed25834fe1', metadata={'source': 'jBEOdMKAkEU'}, page_content="from Chinese automaker BYD. Badl looking car. It sort of looks like a Corolla or maybe a Kia Optima. Vehicles that are ready that are up to snuff in terms of quality, fit and finish, design, that remains to be seen yet. >> You don't see them at all as a competitor. >> No, I I don't I don't think they have a great product. In fact, if you look closely, while Tesla had Elon Musk, the visionary, charismatic billionaire founder, BYD was led by a humble man, a former government employee with no"),
 Document(id='64bf8d7a-adb5-4912-b3ef-8836b45a7d1e', metadata={'source': 'jBEOdMKAkEU'}, page_content='why if you look at this chart, out of the top six selling BEVs and PHEVs, five of them are BYD and just one is Tesla. And more importantly, [music] the sale of the top two PHEVs is more than the sale of the top three EVs combined. This is how Wang solved for cost and accessibility using R&D and 

In [97]:
from langchain_core.prompts import PromptTemplate
prompt = ChatPromptTemplate.from_template(
    template = """You are a helpful assistant. Answer the questions based on the context provided only. Every correct answer will be rewarded
    with 1000 points and every wrong answer will be penalised. Don't try to come up with random answers, only answer the queries 
    from the provided context. If you don't find the answer to a question, just say you don't know.
    <context>
    {context}
    </context>
    question: {input}""",
    # input_variables=['res', 'query']
)
# resp = prompt.invoke({'context':res, 'query':query})

In [44]:
model = ChatOllama(model='qwen2.5:3b')
# response = model.invoke(resp)
# response.content

In [98]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
retriever = db.as_retriever()
parser = StrOutputParser()
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)
chain = (
    {'context': retriever|format_docs, 'input': RunnablePassthrough()} | prompt | model | parser
)
chain.invoke(query)
# print({'input':query})
# final_response.content

"BYD was led by a former government employee with no prior experience as a visionary or charismatic founder. The details provided do not specify who founded BYD; they only mention Wang Song Fu (Wong Chan Fu) as the current president and chairman. Therefore, I don't have enough information to identify the founder of BYD specifically from this context."